## SAT encoding and solver pipeline

This chapter covers the implementation that turns the DNA word-design rules into a SAT instance, runs the Glucose solver, and translates its model back into a readable set of words. The cells follow that pipeline in execution order: define the constraints, generate candidate words and variables, build the CNF clauses, call the solver, and decode the result.

### Dependencies and shared configuration

The implementation uses `Path` to create an absolute path for the generated DIMACS file and `subprocess` to communicate with WSL and Glucose. Both modules are part of Python's standard library, so they do not need to be installed separately.

The following software is needed to run the complete notebook:

- [Python 3.10 or newer](https://www.python.org/downloads/).
- [Windows Subsystem for Linux](https://learn.microsoft.com/windows/wsl/install/) with a Linux distribution such as Ubuntu. The solver function directly calls `wsl.exe` and `wslpath`, so the current implementation is intended to run on Windows with WSL enabled.
- [Glucose 4.2.1](https://github.com/audemard/glucose/releases/tag/4.2.1), downloaded and compiled inside the WSL distribution. This is the official Glucose 4.2 release used by the project.
- The Linux build tools needed to compile Glucose: a C++ compiler, GNU Make, and the zlib development library. On Ubuntu these are provided by the `build-essential` and `zlib1g-dev` packages. Glucose's official instructions specifically require `make` and the zlib library.

No additional Python libraries are used by the implementation itself. After Glucose has been compiled, `solverPath` must contain its absolute Linux path, for example `/home/USER/glucose/simp/glucose_release`.

`ALPHABET` lists the four DNA bases from which every candidate word is built, while `WORD_LENGTH` sets the expected length of each word. 
Both constants are used throughout the later cells. In particular, changing `WORD_LENGTH` also requires reviewing rules that currently assume an eight-letter word, such as the fixed requirement of four `C` or `G` bases.

In [1]:
from pathlib import Path
import subprocess

ALPHABET : list[str] = ['A', 'C', 'G', 'T']
WORD_LENGTH : int = 8

---

### Checking the GC content

`Check_CG_Count` applies the composition rule to one candidate word. It counts every occurrence of `C` and `G` and accepts the word only when their combined count is exactly four. With the configured word length of eight, this means that half of the word must consist of `C` or `G`, while the other half consists of `A` or `T`.

The function returns a Boolean value and does not validate the word's length or alphabet. Those properties are guaranteed later when words are generated from the shared configuration.

In [2]:
def Check_CG_Count(word : str) -> bool:
    """checks the letter count rule for one word"""
    if(word.count('C') + word.count('G') == 4):
        return True
    return False

---

### Checking positional distance

`Check_4_Position_Difference` measures the Hamming distance between two words: it compares bases at the same index and counts how many positions differ. The pair is valid when at least four positions are different. This prevents two selected DNA words from being too similar.

The comparison covers exactly `WORD_LENGTH` positions, so both inputs are expected to contain at least that many characters and to use the same configured word length.

In [3]:
def Check_4_Position_Difference(word_x : str, word_y : str) -> bool:
    """checks the 4-pos rule for 2 words"""
    diffCount = 0
    for i in range(WORD_LENGTH):
        if(word_x[i] != word_y[i]):
            diffCount += 1

    if(diffCount >= 4):
        return True
    return False

---

### Reverse and Watson-Crick complement rules

This block provides the transformations needed for the reverse-complement constraint. `Get_Reverse_Word` reverses the order of the bases, and `Get_WCC_Word` replaces each base with its Watson-Crick complement (`A` with `T`, and `C` with `G`).

`Check_RC_Difference` applies these helpers to a pair of words and then reuses the positional-distance check. It accepts the pair only when the reverse of the first word differs from the complement of the second word in at least four positions. Calling it with the same word twice also provides the self reverse-complement check used during candidate generation. Inputs are expected to contain only bases from `ALPHABET`; another character would not have an entry in the complement mapping.

In [4]:
def Check_RC_Difference(word_x : str, word_y :str) -> bool:
    """checks reverse vs WCC difference rule for 2 words"""
    R_word = Get_Reverse_Word(word_x)
    C_word = Get_WCC_Word(word_y)

    if(Check_4_Position_Difference(R_word, C_word)):
        return True
    return False

def Get_Reverse_Word(word : str) -> str:
    """returns reverse of a word"""
    return word[::-1]

def Get_WCC_Word(word: str) -> str:
    """returns a word WCC-ed counterpart"""
    complements = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C',
    }

    return ''.join(complements[letter] for letter in word)

---

### Generating candidates and SAT variables

`GenerateAllEligibleWordsHelper` extends every partial word with every base in `ALPHABET`. Starting with the empty string and repeating this step `WORD_LENGTH` times makes `GenerateAllEligibleWords` enumerate all possible words of the configured length. It then keeps only words that satisfy both the GC-count rule and the self reverse-complement distance rule.

`GenerateVariablesForWords` assigns a positive integer to each remaining word. These integers are the propositional variable identifiers used in DIMACS CNF: a variable is true when its corresponding word is selected for the final set. Numbering starts at 1 because DIMACS reserves 0 as the end-of-clause marker. The generation order is deterministic, so the same word list produces the same mapping.

Runtime warning: with `WORD_LENGTH = 8`, the first call to `GenerateAllEligibleWords` constructs and filters $4^8 = 65{,}536$ words and returns 17,440 eligible candidates. The cell does not print progress while it runs. Increasing `WORD_LENGTH` makes this enumeration grow exponentially in both time and memory because the number of possible words is $4^{\text{WORD_LENGTH}}$.

In [5]:
def GenerateAllEligibleWords() -> list[str]:
    """returns a list of all words that dont break self-RC and CG-count rules"""
    words = [""]
    for i in range(WORD_LENGTH):
        words = GenerateAllEligibleWordsHelper(words)

    return [word for word in words if(Check_CG_Count(word) and Check_RC_Difference(word, word))]

def GenerateAllEligibleWordsHelper(array : list[str]) -> list[str]:
    """helper for creating every possible word creatable form the alphabet"""
    out = []
    for subword in array:
        for i in ALPHABET:
            out.append(subword + i)
    return out

def GenerateVariablesForWords(words : list[str]) -> dict[str , int]:
    """appoints each word a number and returns a dictionary of the mapping"""
    return {word: index for index, word in enumerate(words, start = 1)}

---

### Encoding pairwise compatibility

`GetRuleClauses` compares every pair of different eligible words. A pair is compatible only when it passes both the direct positional-distance rule and the reverse-complement distance rule. If either rule fails, the function adds the clause `(¬x ∨ ¬y)`, represented as `(-wordVariables[word_x], -wordVariables[word_y])`. This clause prevents both incompatible words from being selected at the same time.

The nested loops visit ordered pairs, so the resulting list contains both orientations of each incompatibility. They express the same restriction and are logically redundant, but remain valid CNF input. Comparisons of a word with itself are skipped because the relevant self constraint was already enforced while generating candidates.

In [6]:
def GetRuleClauses(words : list[str], wordVariables : dict[str , int]) -> list[tuple[int, int]]:
    """generates all CNF clauses for enforcing the problem rules"""

    clauses = []
    for word_x in words:
        for word_y in words:
            if(word_x == word_y):
                continue
            if(not (Check_4_Position_Difference(word_x, word_y) and Check_RC_Difference(word_x, word_y))):
            # only append if words break the rules to enforce they cannot be true at the same time
                clauses.append((-wordVariables[word_x], -wordVariables[word_y]))

    return clauses

---

### Encoding an exact set size

`GetSizeClauses` adds the cardinality constraint that exactly `targetSize` word variables must be true. It uses a dynamic-programming encoding instead of listing every possible combination. The auxiliary state `dp[i][j]` means that at least `j` words have been selected among the first `i` candidates. Each state receives its own SAT variable, and four clauses encode its equivalence to the previous states and the current word variable.

A dedicated positive unit literal represents true; its negation is reused wherever a false initial state is needed. The final two unit clauses require at least `targetSize` selected words and forbid `targetSize + 1`, which together enforce equality. Auxiliary identifiers are allocated after the word variables so the two groups cannot overlap. The function returns both the generated clauses and the largest variable identifier, which is required by the DIMACS header.

In [7]:
def GetSizeClauses(wordVariables : dict[str, int], targetSize : int) -> tuple[list[tuple[int, ...]], int]:
    """generates all CNF clauses for enforcing the "exactly k words in set" rule"""
    clauses = []
    selectedVariables = list(wordVariables.values())
    wordCount = len(selectedVariables)
    nextVariable = max(selectedVariables, default = 0) + 1

    # resereve variable for tautology
    trueLiteral = nextVariable
    nextVariable += 1
    clauses.append((trueLiteral,))

    # reserve variable for contradiction
    falseLiteral = -trueLiteral

    # 2D dynamic list of steps
    # dp[i][j] means "atleast j words were selected from the first i words"
    # initial state is false everywhere
    # targetSize + 2 so the encoding can prevent selecting more than targetSize words  
    dp = [[falseLiteral] * (targetSize + 2) for _ in range(wordCount + 1)]

    # set baseline "atleast 0 words in set" as always true
    for i in range(wordCount + 1):
        dp[i][0] = trueLiteral


    for i in range(1, wordCount + 1):
        selectedVariable = selectedVariables[i - 1]

        for j in range(1, targetSize + 2):
            # Create a new auxiliary variable for dp[i][j].
            currentState = nextVariable
            nextVariable += 1
            dp[i][j] = currentState

            # A: at least j words were already selected before word i.
            alreadyReached = dp[i - 1][j]

            # B: at least j - 1 words were selected before word i.
            previousLevel = dp[i - 1][j - 1]

            # main rule is:
            # currentState <-> alreadyReached OR (previousLevel AND selectedVariable)

            # alreadyReached -> currentState
            clauses.append((-alreadyReached, currentState))

            # previousLevel AND selectedVariable -> currentState
            clauses.append((-previousLevel, -selectedVariable, currentState))

            # currentState -> alreadyReached OR (previousLevel AND selectedVariable)
            clauses.append((-currentState, alreadyReached, previousLevel))
            clauses.append((-currentState, alreadyReached, selectedVariable))

    # Require at least targetSize selected words.
    clauses.append((dp[wordCount][targetSize],))

    # Forbid targetSize + 1 or more selected words.
    # Together, these unit clauses require exactly targetSize words.
    clauses.append((-dp[wordCount][targetSize + 1],))

    return clauses, nextVariable - 1


---

### Writing DIMACS and running Glucose

`Glucose` serializes the complete formula to `glucoseClauses.txt` in DIMACS CNF format. The header records the number of variables and clauses, and every following line contains one clause followed by the required `0` terminator. Running this function replaces the contents of that file with the current formula.

The solver is expected to run inside WSL. The function first asks `wslpath` to translate the Windows file path into a Linux path, then launches the executable supplied through `solverPath` with model output enabled and normal verbosity disabled. The returned `CompletedProcess` contains Glucose's exit code, standard output, and standard error for the parsing stage. A working WSL installation and a valid Linux path to Glucose 4.2 are therefore runtime requirements.

In [8]:
def Glucose(variableCount : int, clauses : list[tuple[int, ...]], solverPath : str) -> subprocess.CompletedProcess[str]:
    """writes all clauses into a file in the correct glucose format and runs glucose on it"""
    inputPath = Path("glucoseClauses.txt").resolve()
    with inputPath.open('w') as file:
        file.write(f"p cnf {variableCount} {len(clauses)}\n")
        for clause in clauses:
            file.write(f"{' '.join(map(str, clause))} 0\n")

    # required conversion of path from virtual machine linux running glucose
    pathConversion = subprocess.run(
        ["wsl.exe", "--exec", "wslpath", "-a", "-u", str(inputPath)],
        capture_output=True,
        text=True,
        check=True,
    )
    wslInputPath = pathConversion.stdout.strip()

    return subprocess.run(
        ["wsl.exe", "--exec", solverPath, "-model", "-verb=0", wslInputPath],
        capture_output=True,
        text=True,
    )


---

### Parsing the solver model

`ParseGlucoseModel` converts Glucose's text output into a set of variables assigned true. Exit code `20` is treated as an unsatisfiable result and produces `None`. For a model, the parser reads lines beginning with `v`, stops at each DIMACS `0` terminator, and collects only positive literals because those are the selected variables. Negative literals describe variables assigned false and are not needed during decoding.

The returned set can contain both word variables and the auxiliary variables introduced by the size encoding. They are separated in the next stage by looking values up in the word-to-variable mapping.

In [9]:
def ParseGlucoseModel(result: subprocess.CompletedProcess[str]) -> set[int] | None:
    """parses model and outcome from glucose output"""
    if result.returncode == 20:
        return None

    selectedVariables = set()

    for line in result.stdout.splitlines():
        tokens = line.split()

        if(not tokens or tokens[0] != 'v'):
            continue

        for token in tokens[1:]:
            literal = int(token)

            if(literal == 0):
                break

            if(literal > 0):
                selectedVariables.add(literal)

    return selectedVariables

---

### Decoding the selected words

`DecodeGlucoseOutput` turns the parsed SAT model into user-facing output. An unsatisfiable model produces a short failure message. Otherwise, the function uses the shared `wordVariables` mapping to keep only true variables that represent actual DNA words, automatically ignoring all auxiliary variables.

Before printing a solution, the function checks that the decoded word count matches the requested set size. A successful result is printed one word per line. This function reports through standard output and returns no separate solution object.

In [10]:
def DecodeGlucoseOutput(parsedModel : set[int] | None, setSize : int) -> None:
    """decodes parsed glucose output into a humanly readable string"""
    if(parsedModel is None):
        return print("Glucose didnt find any solutions.")

    selectedWords = [word for word, variable in wordVariables.items() if variable in parsedModel]
    finalSize = len(selectedWords)

    if(finalSize != setSize):
        return print(f"Couldnt find a model with target size.")
    
    return print(f"Found solution for words:\n{'\n'.join(selectedWords)}")
    

---

### Connecting the solver stages

`Run` is the small orchestration layer for one completed SAT instance. It passes the formula to `Glucose`, parses the returned model, and then decodes that model using the requested set size. Keeping these calls together gives the execution cell a single entry point while leaving formula construction separate from solver execution.

In [11]:
def Run(target_set_size : int, solverPath : str, variableCount : int, clauses : list[tuple[int, ...]]) -> None:
    """connects and runs the whole SAT calculation for a specific set of clauses"""
    glucoseOutput = Glucose(variableCount, clauses, solverPath)
    parsed = ParseGlucoseModel(glucoseOutput)
    DecodeGlucoseOutput(parsed, target_set_size)


---

### Preparing an instance

This cell creates the problem instance used for the next run. By default, `words` contains every candidate accepted by the earlier individual-word rules, although it can be replaced with a custom list. The cell then creates the word-variable mapping and all pairwise compatibility clauses, which depend only on that list.

Two values must be configured for an actual solver run: `solverPath` is the Linux path to the Glucose executable inside WSL, and `set_size` is the exact number of words requested in the solution. The initial values are placeholders, so they should be changed before executing the final cell. Re-running this cell is necessary after changing the word list because both the mapping and the rule clauses must be rebuilt.

Runtime warning: `GetRuleClauses` is the most expensive part of preparing the default instance. It compares every ordered pair of the 17,440 eligible words, which gives 304,136,160 pair checks. The first full preparation can therefore take a long time and use substantial memory without displaying progress. A small custom word list should be used when checking the installation or demonstrating the positive and negative example instances.

In [ ]:
# can be changed for a custom word list
words = GenerateAllEligibleWords()

wordVariables = GenerateVariablesForWords(words)
ruleClauses = GetRuleClauses(words, wordVariables)

# path to solver program
# e.g. "/home/USER/glucose/simp/glucose_release"
solverPath = ""

# 0 <= input set size
set_size = 85

---

### Building and solving the final formula

The final cell builds the clauses that enforce the requested set size, joins them with the pairwise rule clauses, and passes the complete formula to `Run`. `variableCount` includes the original word variables as well as every auxiliary variable created by the cardinality encoding, so the DIMACS header covers the full formula.

Executing this cell writes the CNF file, starts Glucose through WSL, and prints either the decoded word set or a message explaining that no matching solution was found. If `set_size`, `words`, or `solverPath` changes, the preparation cell above should be run again before this one.

Runtime warning: the exact-cardinality encoding grows with the requested set size. For the default 17,440 candidates, increasing `set_size` by one adds 17,440 auxiliary variables and 69,760 clauses. The DIMACS file is written before Glucose starts, so a large target may spend considerable time and memory on formula construction even when the solver later finishes quickly. Solver time itself cannot be predicted from the target size alone and should be measured for every reported instance.

In [13]:
sizeClauses, variableCount = GetSizeClauses(wordVariables, set_size)
clauses = ruleClauses + sizeClauses

print("SAT instance summary")
print(f"  Eligible candidate words: {len(words):,}")
print(f"  Requested set size:       {set_size:,}")
print(f"  Word variables:           {len(wordVariables):,}")
print(f"  Auxiliary variables:      {variableCount - len(wordVariables):,}")
print(f"  Compatibility clauses:    {len(ruleClauses):,}")
print(f"  Cardinality clauses:      {len(sizeClauses):,}")
print(f"  Total CNF clauses:        {len(clauses):,}")
print()

Run(set_size, solverPath, variableCount, clauses)

SAT instance summary
  Eligible candidate words: 17,440
  Requested set size:       85
  Word variables:           17,440
  Auxiliary variables:      1,499,841
  Compatibility clauses:    18,220,192
  Cardinality clauses:      5,999,363
  Total CNF clauses:        24,219,555

Found solution for words:
AAACGGAG
AACATCCG
AACCCTTG
AAGCCAAC
AAGGGCAA
AAGGTGTG
AAGTACCC
AATCCGCA
AATCGCTC
AATGAGGC
AATGGTCG
AATTCCGG
ACACAGGA
ACACCTCT
ACACTCAC
ACAGAACC
ACATGCTG
ACCACCAA
ACCATTGC
ACCCACTT
ACCCTACA
ACCGGTTA
ACCGTGAT
ACCTATCG
ACCTGAAC
ACGAAAGG
ACTACGTG
ACTAGCCT
ACTGCTAC
ACTTTGCC
AGAACCTC
AGAAGACG
AGAGACAG
AGCAAGAC
AGCTACGA
AGGACTAG
AGGGCATA
AGGTGTCA
AGTCATGG
AGTCCCAT
AGTGTCCA
ATATCGCG
ATCCAAGC
ATCGAGCA
ATCGCAAG
ATCGTCTC
ATGCGGTA
CAAACTGC
CACTTGAG
CATAGACC
CCAAACCA
CCAGCAAA
CCGAGTAA
CGCTAATC
CGGCAAAA
CGTAGTTG
CGTATCAC
CGTGAGTA
CGTTCAAG
CTAACCAG
CTGTGATG
CTTTCCTC
GAAAACGG
GAACTGTC
GAAGGACA
GAATGCAC
GACGACTA
GAGTCAGA
GCGAAGTA
GCGTTCAA
GCTCAAAC
GCTGATCA
GTTACCGA
GTTCGTAG
TACAGGCA
TAGCTCCA
TATCGAGG
TCAA

---

## Results

The output of the final code cell first reports the number of candidate words, requested words, propositional variables, auxiliary variables, compatibility clauses, cardinality clauses, and total clauses. These values describe the DIMACS instance constructed for the selected input. With the default generator, 65,536 possible words are reduced to 17,440 candidates by the GC-content and self reverse-complement filters.

If Glucose finds a satisfying assignment, the notebook prints `Found solution for words:` followed by exactly `set_size` words. This list is the human-readable solution required by the exercise. For the default candidate generator, the words satisfy the individual GC-content and self reverse-complement rules, and the SAT clauses guarantee their pairwise compatibility. Custom candidate lists must be checked separately because the notebook does not validate their individual-word constraints.

If Glucose returns an unsatisfiable result, the notebook prints `Glucose didnt find any solutions.` This establishes that no subset of the current candidate list has exactly the requested size. It does not exclude solutions with a smaller target size. The optimization problem can therefore be investigated by running neighbouring target sizes until a satisfiable size and an unsatisfiable size provide lower and upper bounds.

Experiments with the generated set of 17,440 candidates produced the following measured results. Every listed target size was satisfiable. The times are the reported elapsed times for the runs; separate timings for formula construction, DIMACS output, and Glucose were not recorded.

| `set_size` | Result | Elapsed time |
| ---: | --- | ---: |
| 1 | SAT | approximately 2 seconds |
| 10 | SAT | approximately 17 seconds |
| 20 | SAT | approximately 18 seconds |
| 30 | SAT | approximately 20 seconds |
| 64 | SAT | approximately 55 seconds |
| 85 | SAT | approximately 9 minutes |

The increase is not uniform. Runs from `set_size = 10` through `set_size = 30` remained close to 20 seconds, while the increase from 64 to 85 raised the elapsed time from approximately 55 seconds to approximately 9 minutes. Runtime should therefore be measured rather than extrapolated linearly from smaller target sizes. The runs for 64 and 85 both satisfy the exercise requirement for a nontrivial satisfiable instance taking between 10 seconds and 10 minutes.

The saved output of the final code cell now provides the complete witness for `set_size = 85`. It contains 85 distinct words, and an independent check confirmed that every word has length eight and the required GC content, every distinct pair has Hamming distance at least four, and every required reverse/complement comparison also has distance at least four. The same run used 17,440 word variables, 1,499,841 auxiliary variables, 18,220,192 compatibility clauses, and 5,999,363 cardinality clauses, for a total of 1,517,281 variables and 24,219,555 clauses. The full word list is kept in the code-cell output instead of being duplicated in this section.

The small human-readable instances required by the exercise can be demonstrated separately. `words = ["AAAACCCC"]` with `set_size = 1` is the positive instance. `words = ["AAAACCCC", "AAAACCCG"]` with `set_size = 2` is the negative instance because the two words differ in only one position and cannot both be selected. Their executed outputs should be retained in the final notebook.

The notebook is saved with a configured `solverPath`, `set_size = 85`, and the successful decoded output. The approximately nine-minute elapsed time was measured externally rather than by the notebook. The current call also uses `-verb=0`, so detailed Glucose statistics are suppressed, and it always writes `glucoseClauses.txt` instead of providing an option to enable or disable DIMACS output. These points remain limitations relative to the exercise requirements.

---

## Conclusion

The notebook implements the required SAT workflow for the decision version of the DNA word-design problem. It accepts a candidate list and target size, assigns a propositional variable to every candidate, encodes incompatible pairs and the exact set size in CNF, writes the formula in DIMACS format, calls Glucose, and decodes a satisfying model into DNA words. This makes a satisfying result directly checkable against the original problem constraints.

The construction also shows where the full instance becomes expensive. Candidate generation grows exponentially with word length, compatibility construction examines ordered pairs, and the cardinality encoding grows linearly with both the number of candidates and `set_size`. The pairwise stage may also encode the same incompatibility in both orders. These properties increase preparation time, memory consumption, and DIMACS file size before solver time is considered. Comparing each unordered pair once and evaluating a more compact cardinality encoding are possible improvements, but they are not required for the current implementation.

The recorded experiments found satisfiable solutions for every tested target from 1 through 85, with selected intermediate measurements. In particular, `set_size = 64` completed in approximately 55 seconds and `set_size = 85` completed in approximately 9 minutes, so both qualify as nontrivial satisfiable instances under the required runtime interval. These results establish 85 as a demonstrated lower bound on the achievable set size, but they do not show that 85 is optimal because no unsatisfiable larger target was recorded.

The runtime measurements also show that target size alone does not give a reliable linear estimate. The measured time stayed near 20 seconds for target sizes 10, 20, and 30, then increased to 55 seconds for 64 and approximately 9 minutes for 85. This sharp change is consistent with the growing cardinality encoding and a harder SAT search, although separate construction and solver timings would be needed to identify their individual contributions.

The complete decoded set for `set_size = 85` is retained in the notebook output and satisfies all checked constraints, providing a human-readable witness for the strongest recorded result. Detailed Glucose statistics and separate timings for formula construction, DIMACS writing, and solving are still missing, and the small positive and negative instance outputs have not been saved. Within those limitations, the implementation demonstrates the complete conversion from the DNA design constraints to SAT and confirms that valid sets containing at least 85 words exist.